# PagedAttention in a hundred lines
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week4/Paged_KV_Simulator_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 4, Module 7: Serving and the KV cache**

A serving engine does not know how long a request will run. Under **continuous batching**, the scheduler rebuilds the batch every decode step: a finished request leaves, and a waiting one takes its place. If the KV cache manager reserves the maximum length for every request, most of the reserved memory is never used, and fewer requests fit. **PagedAttention** stores the KV cache in fixed-size blocks, 16 tokens each by default in vLLM, and allocates a new block only when a request fills its current one. **Prefix caching** lets requests that start with the same tokens share the same blocks.

In this notebook you simulate a KV cache manager in plain Python. You use the lecture's numbers: a 13B model with 819 KB of KV per token, on an 80 GB H100 with 26 GB of weights. You compare reserving the maximum length with paging, measure the waste, change the block size, and count the blocks prefix caching shares.

**What you will do**
1. Generate a stream of requests with random prompt and output lengths.
2. Simulate continuous batching with contiguous reservations of the maximum length.
3. Simulate the same stream with 16-token blocks.
4. Compare batch size, memory waste and throughput.
5. Change the block size, then simulate prefix caching.

**Before you start:** any runtime works, including CPU. The notebook takes under a minute.

## 1. The memory budget and the requests
The lecture's numbers: 80 GB on the H100, minus 26 GB of weights, leaves 54 GB for the KV cache. At 819 KB per token, that is about 66,000 tokens of KV for all requests together. A 4K-token request holds 3.4 GB, so contiguous reservation fits 16 requests.

Requests arrive at random times, with prompt and output lengths drawn from long-tailed distributions. The scheduler sees the prompt length but not the output length.

In [ ]:
import math
import random
import statistics
from collections import deque

KV_BYTES_PER_TOKEN = 819e3
CAPACITY_TOKENS = int(54e9 / KV_BYTES_PER_TOKEN)
MAX_LEN = 4096
print(f"KV capacity: {CAPACITY_TOKENS:,} tokens")
print(f"Requests that fit with a full {MAX_LEN}-token reservation each: {CAPACITY_TOKENS // MAX_LEN}")


def make_requests(n=600, rate=0.3, seed=0):
    """n requests arriving on average every 1/rate decode steps."""
    rng = random.Random(seed)
    t, reqs = 0.0, []
    for i in range(n):
        t += rng.expovariate(rate)
        prompt = min(int(rng.lognormvariate(math.log(400), 0.8)) + 1, 3000)
        output = min(int(rng.lognormvariate(math.log(200), 0.7)) + 1, MAX_LEN - prompt)
        reqs.append({"id": i, "arrival": t, "prompt": prompt, "output": output})
    return reqs


reqs = make_requests()
print(f"{len(reqs)} requests over {reqs[-1]['arrival']:.0f} steps")
print(f"median prompt {statistics.median(r['prompt'] for r in reqs):.0f} tokens, "
      f"median output {statistics.median(r['output'] for r in reqs):.0f} tokens, "
      f"median total {statistics.median(r['prompt'] + r['output'] for r in reqs):.0f} tokens")

## 2. The simulator
Each step of the loop is one decode step:

1. New arrivals join the waiting queue.
2. The scheduler admits waiting requests in arrival order while their memory fits.
3. Every running request generates one token. Under paging, a request that fills its last block gets a new block.
4. Finished requests leave and free their memory.

`contiguous` reserves `MAX_LEN` tokens per request at admission. `paged` reserves blocks for the prompt plus one token, then one block at a time. If no block is free, the newest running request is **preempted**: its memory is freed and it goes back to the front of the queue, to be recomputed later. vLLM does the same by default.

In [ ]:
def blocks_for(tokens, block):
    return math.ceil(tokens / block) * block


def simulate(reqs, policy, block=16, steps=2500):
    pending = deque(sorted(reqs, key=lambda r: r["arrival"]))
    queue, running = deque(), []
    log = {"batch": [], "reserved": [], "used": []}
    done = generated = preempted = 0

    for t in range(steps):
        while pending and pending[0]["arrival"] <= t:
            queue.append(dict(pending.popleft(), made=0))

        reserved = sum(r["reserved"] for r in running)
        while queue:                                   # admit in arrival order while memory fits
            r = queue[0]
            need = MAX_LEN if policy == "contiguous" else blocks_for(r["prompt"] + 1, block)
            if reserved + need > CAPACITY_TOKENS:
                break
            queue.popleft()
            r["reserved"] = need
            running.append(r)
            reserved += need

        for r in list(running):                        # one decode step for every running request
            if all(x is not r for x in running):
                continue                               # preempted earlier in this step
            length = r["prompt"] + r["made"] + 1
            if policy == "paged" and length > r["reserved"]:
                while reserved + block > CAPACITY_TOKENS and running[-1] is not r:
                    victim = running.pop()             # preempt the newest request
                    reserved -= victim["reserved"]
                    victim["made"] = 0
                    queue.appendleft(victim)
                    preempted += 1
                if reserved + block > CAPACITY_TOKENS:
                    continue                           # nothing left to free: wait one step
                r["reserved"] += block
                reserved += block
            r["made"] += 1
            generated += 1

        for r in [r for r in running if r["made"] >= r["output"]]:
            running.remove(r)
            reserved -= r["reserved"]
            done += 1

        log["batch"].append(len(running))
        log["reserved"].append(reserved)
        log["used"].append(sum(r["prompt"] + r["made"] for r in running))

    used, res = sum(log["used"]), sum(log["reserved"])
    return {"log": log, "done": done, "generated": generated, "preempted": preempted,
            "batch": statistics.mean(log["batch"]), "waste": 1 - used / res if res else 0.0,
            "queue_left": len(queue) + len(pending)}

## 3. Contiguous reservation against paging

In [ ]:
results = {p: simulate(reqs, p) for p in ("contiguous", "paged")}
print(f"{'policy':11} {'mean batch':>10} {'KV waste':>9} {'tokens made':>12} {'finished':>9} {'preempted':>10} {'still waiting':>14}")
for p, r in results.items():
    print(f"{p:11} {r['batch']:10.1f} {r['waste']:9.1%} {r['generated']:12,d} {r['done']:9d} {r['preempted']:10d} {r['queue_left']:14d}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for p, style in (("contiguous", "-"), ("paged", "--")):
    log = results[p]["log"]
    axes[0].plot(log["batch"], style, color="black", label=p)
    axes[1].plot([u / CAPACITY_TOKENS for u in log["used"]], style, color="black", label=p)
axes[0].set_title("running requests (batch size)")
axes[1].set_title("KV memory holding real tokens")
axes[1].set_ylim(0, 1)
for ax in axes:
    ax.set_xlabel("decode step")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Reading the output**
- Contiguous reservation never runs more than 16 requests at once, and most of the memory it reserves holds no tokens: the waste is around 80 to 90 percent, close to the 60 to 80 percent the vLLM paper reports for earlier systems.
- Paging runs several times more requests at once, with waste of a few percent, which is the space left in each request's last block.
- More requests per step means more tokens per weight read, so the same hardware finishes far more requests in the same number of steps. Each request still stores and reads the same KV bytes. Paging recovers capacity, not bandwidth.
- Preemptions appear when paging admits requests that later outgrow the free memory. A real scheduler leaves headroom to limit them.

## 4. Block size
Small blocks waste less space in each request's last block but need a longer block table, with one entry per block. Large blocks need fewer entries but waste more.

In [ ]:
print(f"{'block':>6} {'KV waste':>9} {'mean batch':>11} {'block-table entries per request':>32}")
mean_len = statistics.mean(r["prompt"] + r["output"] for r in reqs)
for block in (1, 16, 64, 256, 1024):
    r = simulate(reqs, "paged", block=block)
    print(f"{block:6d} {r['waste']:9.1%} {r['batch']:11.1f} {mean_len / block:32.0f}")

**Reading the output**
- At 16 tokens the waste is already small. Going to 1 token saves little memory and multiplies the block table by 16.
- At 1024 tokens the waste rises toward the contiguous case.
- The class notes for this module discuss the block size choice further. On a compiled-schedule machine, the block table becomes an indirect gather that the compiler cannot plan ahead, which is why Spyre keeps the number of blocks per request fixed.

## 5. Prefix caching
Many requests start with the same system prompt. A block's KV depends on every token before it, so a block can be shared only if the whole prefix up to that block is identical. The cell builds 10 requests that share a 48-token system prompt, hashes each full block together with its prefix, and counts how many physical blocks are needed.

In [ ]:
BLOCK = 16
rng = random.Random(1)
system_prompt = [rng.randrange(50000) for _ in range(48)]           # 3 full blocks
requests = [system_prompt + [rng.randrange(50000) for _ in range(rng.randrange(20, 60))] for _ in range(10)]
requests.append([12345] + system_prompt)                             # same text, but a timestamp first


def full_block_keys(tokens):
    """One key per full block: the hash of every token up to the end of that block."""
    return [hash(tuple(tokens[: (i + 1) * BLOCK])) for i in range(len(tokens) // BLOCK)]


cache, computed, total = set(), 0, 0
for i, tokens in enumerate(requests):
    keys = full_block_keys(tokens)
    hits = 0
    for key in keys:                     # match from the start, stop at the first miss
        if key in cache:
            hits += 1
        else:
            break
    cache.update(keys)
    total += len(keys)
    computed += len(keys) - hits
    label = "timestamp first" if i == len(requests) - 1 else f"request {i}"
    print(f"{label:16} {len(tokens):3d} tokens, {len(keys)} full blocks, {hits} reused from the cache")

print(f"\nFull blocks requested: {total}, blocks prefilled: {computed}, physical blocks stored: {len(cache)}")

**Reading the output**
- The first request prefills its blocks. Each later request finds the 3 system-prompt blocks in the cache and prefills only its own question.
- The last request has the same system prompt but a different first token, so nothing matches. This is why applications put the part that never changes at the start of the prompt.
- Shared blocks stay in memory after their requests finish, until the cache manager evicts them.

## Try this
1. In section 1, raise `rate` to 0.6. Which policy starts to build a queue first?
2. In section 3, set `CAPACITY_TOKENS` to the KV space of a model with 8 KV heads instead of 40 (five times more tokens). How does the mean batch change?
3. In section 5, change `BLOCK` to 32. How many system-prompt blocks are shared now?

## Summary
- Continuous batching makes the KV memory in use change every step. Reserving the maximum length wastes most of it.
- Paging allocates fixed-size blocks on demand. Waste drops to the last block of each request, and many more requests fit.
- Block size trades waste against the size of the block table.
- Prefix caching shares blocks only for an identical prefix, matched from the first token.

Lecture: Week 4, Module 7, slides 57 to 63 (why serving is a memory problem, inside a serving engine, continuous batching, PagedAttention, prefix caching).